In [ ]:

                                       %%bash
set -e
fusermount -u /content/drive 2>/dev/null || true
umount /content/drive 2>/dev/null || true
rm -rf /content/drive 2>/dev/null || true
echo "✅ Drive unmounted (if it was mounted)."


✅ Drive unmounted (if it was mounted).


In [ ]:
%%bash
set -e
apt-get -y update
apt-get -y install sra-toolkit pigz bowtie2 samtools wget unzip

pip -q install pandas numpy metaphlan


Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,299 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,901 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,468 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Pack

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import os, subprocess, glob, pandas as pd

# ✅ local paths only
BASE = "/content/soil_meta_pipeline"
RAW  = os.path.join(BASE, "raw_fastq")
SRA  = os.path.join(BASE, "sra")
DB   = os.path.join(BASE, "metaphlan_db")
OUT  = os.path.join(BASE, "metaphlan_out")
TMP  = os.path.join(BASE, "tmp_sra")

for p in [BASE, RAW, SRA, DB, OUT, TMP]:
    os.makedirs(p, exist_ok=True)

SRRS_ALL = [
 'SRR12376372','SRR13342225','SRR13396075','SRR13396103','SRR1825760',
 'SRR23183348','SRR23183349','SRR23183368','SRR26201959','SRR26201960',
 'SRR26201961','SRR33853917','SRR33963317','SRR33963318','SRR33963319',
 'SRR33963320','SRR33963321','SRR33963322','SRR33963323','SRR33963324',
 'SRR5365029','SRR8468865','SRR9093167','SRR9830591'
]

# ✅ start with 5 to verify end-to-end; set to 24 after it works
N = 5
SRRS = SRRS_ALL[:N]
print("Running SRRs:", SRRS)

def run(cmd):
    print("RUN:", cmd)
    subprocess.check_call(cmd, shell=True)

!metaphlan --version
!df -h /content


Running SRRs: ['SRR12376372', 'SRR13342225', 'SRR13396075', 'SRR13396103', 'SRR1825760']
MetaPhlAn version 4.2.4 (21 Oct 2025)
No complete MetaPhlAn Bowtie2 database found
Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   22G  205G  10% /


In [ ]:
def fastqs_exist(srr):
    r1 = os.path.join(RAW, f"{srr}_1.fastq.gz")
    r2 = os.path.join(RAW, f"{srr}_2.fastq.gz")
    se = os.path.join(RAW, f"{srr}.fastq.gz")
    return (os.path.exists(se) or (os.path.exists(r1) and os.path.exists(r2)))

for srr in SRRS:
    if fastqs_exist(srr):
        print("SKIP (FASTQ exists):", srr)
        continue

    run(f"prefetch {srr} -O {SRA}")
    run(f"fasterq-dump {SRA}/{srr}/{srr}.sra -O {RAW} --split-files -e 4 --temp {TMP}")
    for fq in glob.glob(f"{RAW}/{srr}*.fastq"):
        run(f"pigz -f {fq}")

print("FASTQ.gz count:", len(glob.glob(RAW + "/*.fastq.gz")))
!ls -lh /content/soil_meta_pipeline/raw_fastq | head -n 30


RUN: prefetch SRR12376372 -O /content/soil_meta_pipeline/sra
RUN: fasterq-dump /content/soil_meta_pipeline/sra/SRR12376372/SRR12376372.sra -O /content/soil_meta_pipeline/raw_fastq --split-files -e 4 --temp /content/soil_meta_pipeline/tmp_sra
RUN: pigz -f /content/soil_meta_pipeline/raw_fastq/SRR12376372_1.fastq
RUN: pigz -f /content/soil_meta_pipeline/raw_fastq/SRR12376372_2.fastq
RUN: prefetch SRR13342225 -O /content/soil_meta_pipeline/sra
RUN: fasterq-dump /content/soil_meta_pipeline/sra/SRR13342225/SRR13342225.sra -O /content/soil_meta_pipeline/raw_fastq --split-files -e 4 --temp /content/soil_meta_pipeline/tmp_sra
RUN: pigz -f /content/soil_meta_pipeline/raw_fastq/SRR13342225_1.fastq
RUN: pigz -f /content/soil_meta_pipeline/raw_fastq/SRR13342225_2.fastq
RUN: prefetch SRR13396075 -O /content/soil_meta_pipeline/sra
RUN: fasterq-dump /content/soil_meta_pipeline/sra/SRR13396075/SRR13396075.sra -O /content/soil_meta_pipeline/raw_fastq --split-files -e 4 --temp /content/soil_meta_pipelin

In [ ]:
# MetaPhlAn v4 uses --db_dir
run(f"metaphlan --install --db_dir {DB}")

print("DB folder contents (first lines):")
!ls -lh /content/soil_meta_pipeline/metaphlan_db | head -n 30



RUN: metaphlan --install --db_dir /content/soil_meta_pipeline/metaphlan_db
DB folder contents (first lines):
total 34G
-rw-r--r-- 1 root   root   32 Feb 15 08:41 mpa_latest
-rw-rw-r-- 1 164011 9930 6.3G Mar 17  2025 mpa_vJan25_CHOCOPhlAnSGB_202503.1.bt2l
-rw-rw-r-- 1 164011 9930 8.1G Mar 17  2025 mpa_vJan25_CHOCOPhlAnSGB_202503.2.bt2l
-rw-rw-r-- 1 164011 9930 187M Mar 17  2025 mpa_vJan25_CHOCOPhlAnSGB_202503.3.bt2l
-rw-rw-r-- 1 164011 9930 4.1G Mar 17  2025 mpa_vJan25_CHOCOPhlAnSGB_202503.4.bt2l
-rw-r--r-- 1 root   root 2.1M Feb 15 09:45 mpa_vJan25_CHOCOPhlAnSGB_202503.nwk
-rw-r--r-- 1 164011 9927 157M Jul 14  2025 mpa_vJan25_CHOCOPhlAnSGB_202503.pkl
-rw-rw-r-- 1 164011 9930 6.3G Mar 17  2025 mpa_vJan25_CHOCOPhlAnSGB_202503.rev.1.bt2l
-rw-rw-r-- 1 164011 9930 8.1G Mar 17  2025 mpa_vJan25_CHOCOPhlAnSGB_202503.rev.2.bt2l
-rw-rw-r-- 1 164011 9927 148K Jun 18  2025 mpa_vJan25_CHOCOPhlAnSGB_202503_VINFO.csv
-rw-r--r-- 1 root   root 844M Feb 15 09:46 mpa_vJan25_CHOCOPhlAnSGB_202503_VSG.fna


In [ ]:
!which bowtie2
!which bowtie2-build
!bowtie2 --version | head -n 2

!ls -lh /content/soil_meta_pipeline/raw_fastq | head -n 20


/usr/bin/bowtie2
/usr/bin/bowtie2-build
/usr/bin/bowtie2-align-s version 2.4.4
64-bit
total 3.4G
-rw-r--r-- 1 root root 1.6G Feb 15 08:18 SRR12376372_1.fastq.gz
-rw-r--r-- 1 root root 1.6G Feb 15 08:18 SRR12376372_2.fastq.gz
-rw-r--r-- 1 root root  35M Feb 15 08:37 SRR13342225_1.fastq.gz
-rw-r--r-- 1 root root  36M Feb 15 08:37 SRR13342225_2.fastq.gz
-rw-r--r-- 1 root root  19M Feb 15 08:38 SRR13396075_1.fastq.gz
-rw-r--r-- 1 root root  20M Feb 15 08:38 SRR13396075_2.fastq.gz
-rw-r--r-- 1 root root  34M Feb 15 08:38 SRR13396103_1.fastq.gz
-rw-r--r-- 1 root root  38M Feb 15 08:38 SRR13396103_2.fastq.gz
-rw-r--r-- 1 root root  12M Feb 15 08:38 SRR1825760.fastq.gz


In [ ]:
import os, subprocess

BASE = "/content/soil_meta_pipeline"
RAW  = os.path.join(BASE, "raw_fastq")
DB   = os.path.join(BASE, "metaphlan_db")
OUT  = os.path.join(BASE, "metaphlan_out")
TMP  = os.path.join(BASE, "tmp_metaphlan")

os.makedirs(OUT, exist_ok=True)
os.makedirs(TMP, exist_ok=True)

srr = "SRR12376372"
r1 = os.path.join(RAW, f"{srr}_1.fastq.gz")
r2 = os.path.join(RAW, f"{srr}_2.fastq.gz")
out_profile = os.path.join(OUT, f"{srr}_profile.tsv")

cmd = (
    f"metaphlan --input_type fastq "
    f"--db_dir {DB} "
    f"-x mpa_vJan25_CHOCOPhlAnSGB_202503 "
    f"-1 {r1} -2 {r2} "
    f"--nproc 4 --tmp_dir {TMP} "
    f"-o {out_profile}"
)

print("RUNNING:\n", cmd)
p = subprocess.run(cmd, shell=True, text=True, capture_output=True)

print("\n===== STDOUT (first 2000 chars) =====\n", p.stdout[:2000])
print("\n===== STDERR (first 4000 chars) =====\n", p.stderr[:4000])
print("\nReturn code:", p.returncode)
print("\nOutput exists?", os.path.exists(out_profile), out_profile)


RUNNING:
 metaphlan --input_type fastq --db_dir /content/soil_meta_pipeline/metaphlan_db -x mpa_vJan25_CHOCOPhlAnSGB_202503 -1 /content/soil_meta_pipeline/raw_fastq/SRR12376372_1.fastq.gz -2 /content/soil_meta_pipeline/raw_fastq/SRR12376372_2.fastq.gz --nproc 4 --tmp_dir /content/soil_meta_pipeline/tmp_metaphlan -o /content/soil_meta_pipeline/metaphlan_out/SRR12376372_profile.tsv

===== STDOUT (first 2000 chars) =====
 

===== STDERR (first 4000 chars) =====
 Sun Feb 15 10:57:41 2026: [Error] You specified forward and reverse reads, but did not specify --subsampling_paired. Provide the input as standard input or with -inp if you don't want to subsample the reads with --subsampling_paired
Sun Feb 15 10:57:41 2026: Stop execution.


Return code: 1

Output exists? False /content/soil_meta_pipeline/metaphlan_out/SRR12376372_profile.tsv


In [ ]:
import os, subprocess

BASE = "/content/soil_meta_pipeline"
RAW  = os.path.join(BASE, "raw_fastq")
DB   = os.path.join(BASE, "metaphlan_db")
OUT  = os.path.join(BASE, "metaphlan_out")
TMP  = os.path.join(BASE, "tmp_metaphlan")
MAP  = os.path.join(BASE, "metaphlan_mapout")

os.makedirs(OUT, exist_ok=True)
os.makedirs(TMP, exist_ok=True)
os.makedirs(MAP, exist_ok=True)

srr = "SRR12376372"
r1 = os.path.join(RAW, f"{srr}_1.fastq.gz")
r2 = os.path.join(RAW, f"{srr}_2.fastq.gz")

out_profile = os.path.join(OUT, f"{srr}_profile.tsv")
mapout_file = os.path.join(MAP, f"{srr}.bowtie2.bz2")

cmd = (
    f"metaphlan {r1},{r2} --input_type fastq "
    f"--db_dir {DB} -x mpa_vJan25_CHOCOPhlAnSGB_202503 "
    f"--nproc 4 --tmp_dir {TMP} "
    f"--mapout {mapout_file} "
    f"-o {out_profile}"
)

print("RUNNING:\n", cmd)
p = subprocess.run(cmd, shell=True, text=True, capture_output=True)

print("\n===== STDERR (first 2000 chars) =====\n", p.stderr[:2000])
print("\nReturn code:", p.returncode)
print("Output exists?", os.path.exists(out_profile), out_profile)



RUNNING:
 metaphlan /content/soil_meta_pipeline/raw_fastq/SRR12376372_1.fastq.gz,/content/soil_meta_pipeline/raw_fastq/SRR12376372_2.fastq.gz --input_type fastq --db_dir /content/soil_meta_pipeline/metaphlan_db -x mpa_vJan25_CHOCOPhlAnSGB_202503 --nproc 4 --tmp_dir /content/soil_meta_pipeline/tmp_metaphlan --mapout /content/soil_meta_pipeline/metaphlan_mapout/SRR12376372.bowtie2.bz2 -o /content/soil_meta_pipeline/metaphlan_out/SRR12376372_profile.tsv

===== STDERR (first 2000 chars) =====
 Killed
Killed


Return code: 137
Output exists? False /content/soil_meta_pipeline/metaphlan_out/SRR12376372_profile.tsv


In [ ]:
import os, subprocess

BASE="/content/soil_meta_pipeline"
RAW=os.path.join(BASE,"raw_fastq")
DB=os.path.join(BASE,"metaphlan_db")
OUT=os.path.join(BASE,"metaphlan_out")
TMP=os.path.join(BASE,"tmp_metaphlan")
MAP=os.path.join(BASE,"metaphlan_mapout")

os.makedirs(OUT, exist_ok=True)
os.makedirs(TMP, exist_ok=True)
os.makedirs(MAP, exist_ok=True)

srr="SRR13396075"   # smaller one
r1=os.path.join(RAW,f"{srr}_1.fastq.gz")
r2=os.path.join(RAW,f"{srr}_2.fastq.gz")
out_profile=os.path.join(OUT,f"{srr}_profile.tsv")
mapout_file=os.path.join(MAP,f"{srr}.bowtie2.bz2")

cmd=(
    f"metaphlan {r1},{r2} --input_type fastq "
    f"--db_dir {DB} -x mpa_vJan25_CHOCOPhlAnSGB_202503 "
    f"--nproc 1 --tmp_dir {TMP} "          # ✅ 1 thread = low RAM
    f"--mapout {mapout_file} "
    f"-o {out_profile}"
)

print("RUN:", cmd)
p=subprocess.run(cmd, shell=True, text=True, capture_output=True)
print("Return code:", p.returncode)
print("STDERR head:\n", p.stderr[:1200])
print("Output exists:", os.path.exists(out_profile), out_profile)


RUN: metaphlan /content/soil_meta_pipeline/raw_fastq/SRR13396075_1.fastq.gz,/content/soil_meta_pipeline/raw_fastq/SRR13396075_2.fastq.gz --input_type fastq --db_dir /content/soil_meta_pipeline/metaphlan_db -x mpa_vJan25_CHOCOPhlAnSGB_202503 --nproc 1 --tmp_dir /content/soil_meta_pipeline/tmp_metaphlan --mapout /content/soil_meta_pipeline/metaphlan_mapout/SRR13396075.bowtie2.bz2 -o /content/soil_meta_pipeline/metaphlan_out/SRR13396075_profile.tsv
Return code: 137
STDERR head:
 Killed
Killed

Output exists: False /content/soil_meta_pipeline/metaphlan_out/SRR13396075_profile.tsv


In [ ]:
!free -h


               total        used        free      shared  buff/cache   available
Mem:            12Gi       680Mi        11Gi        24Mi       746Mi        11Gi
Swap:             0B          0B          0B


In [ ]:
%%bash
set -e
apt-get -y update
apt-get -y install kraken2 wget unzip pigz
pip -q install pandas numpy


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Fetched 384 kB in 1s (285 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
pigz is already the newest version (2.6-1).
unzip is already the newest version (6.0-26ubuntu3.2).
wget is already the newest version (1.21.2-2ubuntu1.1).
The following additional packages will be installed:
  ncbi-blast+ ncbi-data
The followin

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
set -e
mkdir -p /content/soil_meta_pipeline/kraken_db
cd /content/soil_meta_pipeline/kraken_db

apt-get -y update >/dev/null
apt-get -y install -qq curl ca-certificates >/dev/null

rm -f minikraken2_v2_8GB_201904.tgz

# ✅ WORKING mirror (AWS public dataset)
curl -L --retry 8 --retry-delay 5 -o minikraken2_v2_8GB_201904.tgz \
  https://genome-idx.s3.amazonaws.com/kraken/minikraken2_v2_8GB_201904.tgz

ls -lh minikraken2_v2_8GB_201904.tgz

tar -xzf minikraken2_v2_8GB_201904.tgz

echo "✅ DB folder:"
ls -lh | head -n 40


-rw-r--r-- 1 root root 5.6G Feb 15 11:39 minikraken2_v2_8GB_201904.tgz
✅ DB folder:
total 5.6G
-rw-r--r-- 1 root root 5.6G Feb 15 11:39 minikraken2_v2_8GB_201904.tgz
drwxrwsr-x 2 3481 5042 4.0K Apr 23  2019 minikraken2_v2_8GB_201904_UPDATE
-rw-r--r-- 1 root root    0 Feb 15 11:35 minikraken.tgz


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5661M  100 5661M    0     0  49.3M      0  0:01:54  0:01:54 --:--:-- 49.6M


In [ ]:
%%bash
set -e
rm -f /content/soil_meta_pipeline/kraken_db/minikraken.tgz

echo "DB files (should include hash.k2d / opts.k2d / taxo.k2d):"
ls -lh /content/soil_meta_pipeline/kraken_db/minikraken2_v2_8GB_201904_UPDATE | head -n 50


DB files (should include hash.k2d / opts.k2d / taxo.k2d):
total 7.5G
-rw-rw-r-- 1 3481 5042 2.1M Apr 23  2019 database100mers.kmer_distrib
-rw-rw-r-- 1 3481 5042 1.9M Apr 23  2019 database150mers.kmer_distrib
-rw-rw-r-- 1 3481 5042 1.7M Apr 23  2019 database200mers.kmer_distrib
-rw-rw-r-- 1 3481 5042 7.5G Apr 23  2019 hash.k2d
-rw-rw-r-- 1 3481 5042   56 Apr 23  2019 opts.k2d
-rw-rw-r-- 1 3481 5042 1.7M Apr 23  2019 taxo.k2d


In [ ]:
import os, glob, subprocess

BASE="/content/soil_meta_pipeline"
RAW=os.path.join(BASE,"raw_fastq")

# ✅ IMPORTANT: point to the extracted DB folder
KDB=os.path.join(BASE,"kraken_db","minikraken2_v2_8GB_201904_UPDATE")

OUT=os.path.join(BASE,"kraken_out")
os.makedirs(OUT, exist_ok=True)

def run(cmd):
    print("RUN:", cmd)
    subprocess.check_call(cmd, shell=True)

# detect SRRs present
samples=set()
for fp in glob.glob(RAW + "/*.fastq.gz"):
    bn=os.path.basename(fp)
    if bn.endswith("_1.fastq.gz"):
        samples.add(bn.replace("_1.fastq.gz",""))
    elif bn.endswith(".fastq.gz") and not bn.endswith("_2.fastq.gz"):
        samples.add(bn.replace(".fastq.gz",""))

samples=sorted(samples)
print("Detected samples:", samples)

for srr in samples:
    report = os.path.join(OUT, f"{srr}.report")
    out    = os.path.join(OUT, f"{srr}.kraken")
    if os.path.exists(report):
        print("SKIP (report exists):", srr)
        continue

    r1=os.path.join(RAW,f"{srr}_1.fastq.gz")
    r2=os.path.join(RAW,f"{srr}_2.fastq.gz")
    se=os.path.join(RAW,f"{srr}.fastq.gz")

    if os.path.exists(r1) and os.path.exists(r2):
        cmd = f"kraken2 --db {KDB} --threads 2 --paired {r1} {r2} --report {report} --output {out}"
    else:
        cmd = f"kraken2 --db {KDB} --threads 2 {se} --report {report} --output {out}"

    run(cmd)

print("✅ Reports made:", len(glob.glob(OUT + "/*.report")))
!ls -lh /content/soil_meta_pipeline/kraken_out | head -n 30


Detected samples: ['SRR12376372', 'SRR13342225', 'SRR13396075', 'SRR13396103', 'SRR1825760']
RUN: kraken2 --db /content/soil_meta_pipeline/kraken_db/minikraken2_v2_8GB_201904_UPDATE --threads 2 --paired /content/soil_meta_pipeline/raw_fastq/SRR12376372_1.fastq.gz /content/soil_meta_pipeline/raw_fastq/SRR12376372_2.fastq.gz --report /content/soil_meta_pipeline/kraken_out/SRR12376372.report --output /content/soil_meta_pipeline/kraken_out/SRR12376372.kraken
RUN: kraken2 --db /content/soil_meta_pipeline/kraken_db/minikraken2_v2_8GB_201904_UPDATE --threads 2 --paired /content/soil_meta_pipeline/raw_fastq/SRR13342225_1.fastq.gz /content/soil_meta_pipeline/raw_fastq/SRR13342225_2.fastq.gz --report /content/soil_meta_pipeline/kraken_out/SRR13342225.report --output /content/soil_meta_pipeline/kraken_out/SRR13342225.kraken
RUN: kraken2 --db /content/soil_meta_pipeline/kraken_db/minikraken2_v2_8GB_201904_UPDATE --threads 2 --paired /content/soil_meta_pipeline/raw_fastq/SRR13396075_1.fastq.gz /con

In [ ]:
import os, glob
import pandas as pd

OUT="/content/soil_meta_pipeline/kraken_out"
reports=sorted(glob.glob(OUT + "/*.report"))
print("Reports found:", len(reports))

tables=[]
for rp in reports:
    sample=os.path.basename(rp).replace(".report","")
    df=pd.read_csv(rp, sep="\t", header=None,
                   names=["pct","clade_reads","taxon_reads","rank","taxid","name"])
    df["feature"] = df["rank"].astype(str) + "|" + df["name"].astype(str).str.strip()
    tables.append(df.set_index("feature")["pct"].rename(sample))

X = pd.concat(tables, axis=1).fillna(0.0).T
X.index.name="sample"
X.reset_index(inplace=True)

out_csv="/content/soil_meta_pipeline/taxonomy_kraken2_minikraken.csv"
X.to_csv(out_csv, index=False)

print("✅ Saved:", out_csv, "shape:", X.shape)
X.head()


Reports found: 5
✅ Saved: /content/soil_meta_pipeline/taxonomy_kraken2_minikraken.csv shape: (5, 7689)


feature,sample,U|unclassified,R|root,R1|cellular organisms,D|Bacteria,P|Proteobacteria,C|Alphaproteobacteria,O|Rhizobiales,F|Bradyrhizobiaceae,G|Bradyrhizobium,...,G|Jeotgalibaca,S|Jeotgalibaca dankookensis,S|Mycoplasma genitalium,S|Parabacteroides distasonis,S1|Parabacteroides distasonis ATCC 8503,S|Candidatus Cardinium hertigii,S|Blattabacterium punctulatus,S|Chlamydia sp. 2742-308,S|Dictyoglomus thermophilum,S1|Dictyoglomus thermophilum H-6-12
0,SRR12376372,94.87,5.13,5.13,4.83,3.01,1.04,0.46,0.15,0.09,...,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.0
1,SRR13342225,1.13,98.87,98.60,98.47,9.36,4.05,1.27,0.04,0.00,...,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.0
2,SRR13396075,0.14,99.86,99.86,99.71,60.48,1.05,0.53,0.00,0.00,...,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.0
3,SRR13396103,1.08,98.92,98.08,97.94,9.20,3.91,1.26,0.04,0.00,...,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.0
4,SRR1825760,1.27,98.73,98.73,98.73,41.98,7.79,1.18,0.14,0.00,...,0.0,0.0,0.01,0.01,0.01,0.0,0.02,0.0,0.0,0.0


In [ ]:
from google.colab import files
files.download("/content/soil_meta_pipeline/taxonomy_kraken2_minikraken.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, subprocess, glob

BASE="/content/soil_meta_pipeline"
RAW=os.path.join(BASE,"raw_fastq")
SRA=os.path.join(BASE,"sra")
TMP=os.path.join(BASE,"tmp_sra")
os.makedirs(RAW, exist_ok=True)
os.makedirs(SRA, exist_ok=True)
os.makedirs(TMP, exist_ok=True)

ALL_SRRS = [
 'SRR12376372','SRR13342225','SRR13396075','SRR13396103','SRR1825760',
 'SRR23183348','SRR23183349','SRR23183368','SRR26201959','SRR26201960',
 'SRR26201961','SRR33853917','SRR33963317','SRR33963318','SRR33963319',
 'SRR33963320','SRR33963321','SRR33963322','SRR33963323','SRR33963324',
 'SRR5365029','SRR8468865','SRR9093167','SRR9830591'
]

def run(cmd):
    print("RUN:", cmd)
    subprocess.check_call(cmd, shell=True)

def fastqs_exist(srr):
    r1=os.path.join(RAW,f"{srr}_1.fastq.gz")
    r2=os.path.join(RAW,f"{srr}_2.fastq.gz")
    se=os.path.join(RAW,f"{srr}.fastq.gz")
    return os.path.exists(se) or (os.path.exists(r1) and os.path.exists(r2))

for srr in ALL_SRRS:
    if fastqs_exist(srr):
        print("SKIP (FASTQ exists):", srr)
        continue
    run(f"prefetch {srr} -O {SRA}")
    run(f"fasterq-dump {SRA}/{srr}/{srr}.sra -O {RAW} --split-files -e 4 --temp {TMP}")
    for fq in glob.glob(f"{RAW}/{srr}*.fastq"):
        run(f"pigz -f {fq}")

print("✅ FASTQ.gz files:", len(glob.glob(RAW + '/*.fastq.gz')))
!ls -lh /content/soil_meta_pipeline/raw_fastq | head -n 30


SKIP (FASTQ exists): SRR12376372
SKIP (FASTQ exists): SRR13342225
SKIP (FASTQ exists): SRR13396075
SKIP (FASTQ exists): SRR13396103
SKIP (FASTQ exists): SRR1825760
RUN: prefetch SRR23183348 -O /content/soil_meta_pipeline/sra
RUN: fasterq-dump /content/soil_meta_pipeline/sra/SRR23183348/SRR23183348.sra -O /content/soil_meta_pipeline/raw_fastq --split-files -e 4 --temp /content/soil_meta_pipeline/tmp_sra
RUN: pigz -f /content/soil_meta_pipeline/raw_fastq/SRR23183348_1.fastq
RUN: pigz -f /content/soil_meta_pipeline/raw_fastq/SRR23183348_2.fastq
RUN: prefetch SRR23183349 -O /content/soil_meta_pipeline/sra
RUN: fasterq-dump /content/soil_meta_pipeline/sra/SRR23183349/SRR23183349.sra -O /content/soil_meta_pipeline/raw_fastq --split-files -e 4 --temp /content/soil_meta_pipeline/tmp_sra
RUN: pigz -f /content/soil_meta_pipeline/raw_fastq/SRR23183349_2.fastq
RUN: pigz -f /content/soil_meta_pipeline/raw_fastq/SRR23183349_1.fastq
RUN: prefetch SRR23183368 -O /content/soil_meta_pipeline/sra
RUN: fa

KeyboardInterrupt: 

In [ ]:
%%bash
set -e
RAW=/content/soil_meta_pipeline/raw_fastq

echo "1) Any uncompressed FASTQ left?"
ls -lh $RAW/*.fastq 2>/dev/null | head || echo "✅ none"

echo "2) Compress remaining .fastq (if any)..."
for f in $RAW/*.fastq; do
  [ -e "$f" ] || continue
  echo "compressing: $f"
  pigz -f "$f"
done

echo "3) Remove zero-byte gz (if any)"
find $RAW -maxdepth 1 -name "*.fastq.gz" -size 0 -print -delete || true

echo "✅ Done. Current files:"
ls -lh $RAW | head -n 40


1) Any uncompressed FASTQ left?
-rw-r--r-- 1 root root 26G Feb 15 15:46 /content/soil_meta_pipeline/raw_fastq/SRR33963317_2.fastq
2) Compress remaining .fastq (if any)...
compressing: /content/soil_meta_pipeline/raw_fastq/SRR33963317_2.fastq
3) Remove zero-byte gz (if any)
✅ Done. Current files:
total 17G
-rw-r--r-- 1 root root 1.6G Feb 15 08:18 SRR12376372_1.fastq.gz
-rw-r--r-- 1 root root 1.6G Feb 15 08:18 SRR12376372_2.fastq.gz
-rw-r--r-- 1 root root  35M Feb 15 08:37 SRR13342225_1.fastq.gz
-rw-r--r-- 1 root root  36M Feb 15 08:37 SRR13342225_2.fastq.gz
-rw-r--r-- 1 root root  19M Feb 15 08:38 SRR13396075_1.fastq.gz
-rw-r--r-- 1 root root  20M Feb 15 08:38 SRR13396075_2.fastq.gz
-rw-r--r-- 1 root root  34M Feb 15 08:38 SRR13396103_1.fastq.gz
-rw-r--r-- 1 root root  38M Feb 15 08:38 SRR13396103_2.fastq.gz
-rw-r--r-- 1 root root  12M Feb 15 08:38 SRR1825760.fastq.gz
-rw-r--r-- 1 root root  13M Feb 15 14:48 SRR23183348_1.fastq.gz
-rw-r--r-- 1 root root  16M Feb 15 14:48 SRR23183348_2.f

In [ ]:
import glob, os

RAW="/content/soil_meta_pipeline/raw_fastq"

pairs=set([os.path.basename(x).replace("_1.fastq.gz","") for x in glob.glob(RAW+"/*_1.fastq.gz")])
pairs2=set([os.path.basename(x).replace("_2.fastq.gz","") for x in glob.glob(RAW+"/*_2.fastq.gz")])
paired_ready=sorted(pairs & pairs2)

singles=set([os.path.basename(x).replace(".fastq.gz","")
             for x in glob.glob(RAW+"/*.fastq.gz")
             if not x.endswith("_1.fastq.gz") and not x.endswith("_2.fastq.gz")])

all_samples=sorted(set(paired_ready) | singles)

print("Paired-ready samples:", len(paired_ready))
print("Single-end samples:", len(singles))
print("TOTAL samples ready:", len(all_samples))
print("First 20:", all_samples[:20])


Paired-ready samples: 12
Single-end samples: 1
TOTAL samples ready: 13
First 20: ['SRR12376372', 'SRR13342225', 'SRR13396075', 'SRR13396103', 'SRR1825760', 'SRR23183348', 'SRR23183349', 'SRR23183368', 'SRR26201959', 'SRR26201960', 'SRR26201961', 'SRR33853917', 'SRR33963317']


In [ ]:
import os, glob, subprocess

BASE="/content/soil_meta_pipeline"
RAW=os.path.join(BASE,"raw_fastq")
KDB=os.path.join(BASE,"kraken_db","minikraken2_v2_8GB_201904_UPDATE")
OUT=os.path.join(BASE,"kraken_out")
os.makedirs(OUT, exist_ok=True)

def run(cmd):
    print("RUN:", cmd)
    subprocess.check_call(cmd, shell=True)

# detect SRRs ready
samples=set()
pairs=set([os.path.basename(x).replace("_1.fastq.gz","") for x in glob.glob(RAW+"/*_1.fastq.gz")])
pairs2=set([os.path.basename(x).replace("_2.fastq.gz","") for x in glob.glob(RAW+"/*_2.fastq.gz")])
paired_ready=sorted(pairs & pairs2)

singles=set([os.path.basename(x).replace(".fastq.gz","")
             for x in glob.glob(RAW+"/*.fastq.gz")
             if not x.endswith("_1.fastq.gz") and not x.endswith("_2.fastq.gz")])

all_samples=sorted(set(paired_ready) | singles)
print("Samples to classify:", len(all_samples))

for srr in all_samples:
    report=os.path.join(OUT,f"{srr}.report")
    out=os.path.join(OUT,f"{srr}.kraken")

    if os.path.exists(report):
        print("SKIP (report exists):", srr)
        continue

    r1=os.path.join(RAW,f"{srr}_1.fastq.gz")
    r2=os.path.join(RAW,f"{srr}_2.fastq.gz")
    se=os.path.join(RAW,f"{srr}.fastq.gz")

    if os.path.exists(r1) and os.path.exists(r2):
        cmd=f"kraken2 --db {KDB} --threads 2 --paired {r1} {r2} --report {report} --output {out}"
    else:
        cmd=f"kraken2 --db {KDB} --threads 2 {se} --report {report} --output {out}"

    run(cmd)

print("✅ Reports:", len(glob.glob(OUT+'/*.report')))
!ls -lh /content/soil_meta_pipeline/kraken_out | head -n 30


Samples to classify: 13
SKIP (report exists): SRR12376372
SKIP (report exists): SRR13342225
SKIP (report exists): SRR13396075
SKIP (report exists): SRR13396103
SKIP (report exists): SRR1825760
RUN: kraken2 --db /content/soil_meta_pipeline/kraken_db/minikraken2_v2_8GB_201904_UPDATE --threads 2 --paired /content/soil_meta_pipeline/raw_fastq/SRR23183348_1.fastq.gz /content/soil_meta_pipeline/raw_fastq/SRR23183348_2.fastq.gz --report /content/soil_meta_pipeline/kraken_out/SRR23183348.report --output /content/soil_meta_pipeline/kraken_out/SRR23183348.kraken
RUN: kraken2 --db /content/soil_meta_pipeline/kraken_db/minikraken2_v2_8GB_201904_UPDATE --threads 2 --paired /content/soil_meta_pipeline/raw_fastq/SRR23183349_1.fastq.gz /content/soil_meta_pipeline/raw_fastq/SRR23183349_2.fastq.gz --report /content/soil_meta_pipeline/kraken_out/SRR23183349.report --output /content/soil_meta_pipeline/kraken_out/SRR23183349.kraken
RUN: kraken2 --db /content/soil_meta_pipeline/kraken_db/minikraken2_v2_8GB_

In [ ]:
import os, glob, pandas as pd

OUT="/content/soil_meta_pipeline/kraken_out"
reports=sorted(glob.glob(OUT+"/*.report"))
print("Reports found:", len(reports))

tables=[]
for rp in reports:
    sample=os.path.basename(rp).replace(".report","")
    df=pd.read_csv(rp, sep="\t", header=None,
                   names=["pct","clade_reads","taxon_reads","rank","taxid","name"])
    df["feature"]=df["rank"].astype(str)+"|"+df["name"].astype(str).str.strip()
    tables.append(df.set_index("feature")["pct"].rename(sample))

X=pd.concat(tables, axis=1).fillna(0.0).T
X.index.name="sample"
X.reset_index(inplace=True)

out_csv="/content/soil_meta_pipeline/taxonomy_kraken2_minikraken.csv"
X.to_csv(out_csv, index=False)
print("✅ Saved:", out_csv, "shape:", X.shape)

from google.colab import files
files.download(out_csv)


Reports found: 13
✅ Saved: /content/soil_meta_pipeline/taxonomy_kraken2_minikraken.csv shape: (13, 9784)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np

INFILE = "taxonomy_kraken2_minikraken.csv"   # change path if needed
df = pd.read_csv(INFILE)

# ----------------------------
# 1) Split sample + features
# ----------------------------
sample_col = "sample"
X = df.drop(columns=[sample_col]).copy()
samples = df[sample_col].copy()

# drop very generic nodes
drop_cols = [c for c in X.columns if c in ["R|root", "R1|cellular organisms"]]
X = X.drop(columns=drop_cols, errors="ignore")

# optional: drop unclassified (try both later)
# X = X.drop(columns=["U|unclassified"], errors="ignore")

# convert percent -> proportion
X = X / 100.0

# ----------------------------
# 2) Collapse to Genus level
# columns look like "G|Bradyrhizobium"
# ----------------------------
genus_cols = [c for c in X.columns if c.startswith("G|")]
# if you already have genus cols, great. But you may also have species cols only; we handle both.

# If species are present, and genus exists too, keep genus only.
# If genus missing but species present, we can map species -> genus by string split:
species_cols = [c for c in X.columns if c.startswith("S|")]

if len(genus_cols) == 0 and len(species_cols) > 0:
    # species -> genus approximation: take first word of species name as genus
    # "S|Bradyrhizobium icense" -> "G|Bradyrhizobium"
    genus_map = {}
    for c in species_cols:
        name = c.split("|", 1)[1].strip()
        genus = name.split()[0]
        genus_map[c] = f"G|{genus}"
    Xg = X[species_cols].rename(columns=genus_map).groupby(axis=1, level=0).sum()
else:
    # if genus exists, just use those (most stable)
    Xg = X[genus_cols].copy()

# ----------------------------
# 3) Filter rare genera
# ----------------------------
# keep genera present (>0) in at least 20% of samples
min_prevalence = 0.20
prevalence = (Xg > 0).mean(axis=0)

# keep genera with mean abundance >= 0.0001 (0.01%)
min_mean_abund = 0.0001
mean_abund = Xg.mean(axis=0)

keep = (prevalence >= min_prevalence) & (mean_abund >= min_mean_abund)
Xg_filt = Xg.loc[:, keep].copy()

print("Genus features before:", Xg.shape[1])
print("Genus features after filter:", Xg_filt.shape[1])

# ----------------------------
# 4) CLR transform (compositional)
# add small pseudocount to avoid log(0)
# ----------------------------
eps = 1e-6
Xp = Xg_filt + eps
gm = np.exp(np.mean(np.log(Xp), axis=1))          # geometric mean per sample
X_clr = np.log(Xp.div(gm, axis=0))

# ----------------------------
# 5) Diversity features
# ----------------------------
def shannon(p):
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def simpson(p):
    return float(1.0 - (p**2).sum())

div = pd.DataFrame({
    "sample": samples,
    "richness_gt_1e-4": (Xg > 1e-4).sum(axis=1),
    "shannon": X.apply(shannon, axis=1),
    "simpson": X.apply(simpson, axis=1),
})

# ----------------------------
# 6) Final ML matrix
# ----------------------------
X_final = pd.concat([samples, X_clr.reset_index(drop=True), div.drop(columns=["sample"])], axis=1)

OUTFILE = "taxonomy_ml_ready_genus_clr.csv"
X_final.to_csv(OUTFILE, index=False)
print("✅ Saved:", OUTFILE, "shape:", X_final.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'taxonomy_kraken2_minikraken.csv'